In [1]:
!pip install pytextrank spacy -q
!python -m spacy download ro_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 110.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ro_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import numpy as np
import re
import spacy
import pytextrank
from sentence_transformers import SentenceTransformer, util as st_util
from IPython.display import display, HTML
from google.colab import drive

/usr/local/lib/python3.12/dist-packages


In [3]:
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/Licenta/medical_dataset_final.csv'

Mounted at /content/drive


In [4]:
df = pd.read_csv(CSV_PATH, sep=';', on_bad_lines='skip')
df = df[['filename', 'rezumat', 'keywords', 'status']].copy()
df['rezumat'] = df['rezumat'].fillna('')

print(f"Date încărcate: {len(df)} articole")

Date încărcate: 1114 articole


In [9]:
print("Se încarcă modelul spaCy pentru română...")
nlp = spacy.load('ro_core_news_sm')
nlp.add_pipe('textrank')  # adaugă TextRank în pipeline-ul spaCy
nlp.get_pipe('textrank').within_sentence = False
print("Model încărcat.")

TOP_N = 5  # număr de keywords extrase per document

Se încarcă modelul spaCy pentru română...
Model încărcat.


In [10]:
def extract_textrank(text, n=TOP_N):
    if not text or len(text.strip()) < 20:
        return []
    doc = nlp(text)
    keywords = [(p.text.lower(), round(p.rank, 4))
                for p in doc._.phrases
                if len(p.text.split()) <= 2][:n]
    return keywords

print("Extrag keywords cu TextRank...")
results = []
for i, row in df.iterrows():
    kw = extract_textrank(row['rezumat'])
    results.append({
        'filename'           : row['filename'],
        'keywords_extracted' : '; '.join([k for k, s in kw]),
        'keywords_original'  : row['keywords'],
    })

results_df = pd.DataFrame(results)
print(f"Keywords extrase pentru {len(results_df)} articole")

Extrag keywords cu TextRank...
Keywords extrase pentru 1114 articole


In [11]:
SIM_THRESHOLD = 0.6
print("\nSe încarcă modelul semantic...")
sem_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def parse_keywords(text):
    if pd.isna(text) or str(text).strip() == '':
        return []
    return [w.strip().lower() for w in re.split(r'[;,]', str(text)) if w.strip()]

def compute_metrics(extracted_str, original_str):
    extracted = parse_keywords(extracted_str)
    original  = parse_keywords(original_str)
    if not extracted or not original:
        return 0.0, 0.0, 0.0

    all_terms  = extracted + original
    embeddings = sem_model.encode(all_terms, convert_to_tensor=True)
    emb_ext    = embeddings[:len(extracted)]
    emb_orig   = embeddings[len(extracted):]

    matched_orig = set()
    tp = 0
    for i, e in enumerate(extracted):
        for j, o in enumerate(original):
            if j in matched_orig:
                continue
            string_match   = e in o or o in e
            cos_score      = st_util.cos_sim(emb_ext[i], emb_orig[j]).item()
            semantic_match = cos_score >= SIM_THRESHOLD
            if string_match or semantic_match:
                tp += 1
                matched_orig.add(j)
                break

    precision = tp / len(extracted)
    recall    = tp / len(original)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

print("Calculez metrici semantice...")
results_df[['precision', 'recall', 'f1']] = results_df.apply(
    lambda r: pd.Series(compute_metrics(r['keywords_extracted'], r['keywords_original'])),
    axis=1
)

avg_p  = results_df['precision'].mean()
avg_r  = results_df['recall'].mean()
avg_f1 = results_df['f1'].mean()

print(f"\n{'='*45}")
print(f"  Precision medie : {avg_p:.4f}  ({avg_p:.2%})")
print(f"  Recall mediu    : {avg_r:.4f}  ({avg_r:.2%})")
print(f"  F1 mediu        : {avg_f1:.4f}  ({avg_f1:.2%})")
print(f"{'='*45}")
print(f"  Documente cu F1 > 0 : {(results_df['f1'] > 0).sum()} / {len(results_df)}")


Se încarcă modelul semantic...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Calculez metrici semantice...

  Precision medie : 0.2652  (26.52%)
  Recall mediu    : 0.3165  (31.65%)
  F1 mediu        : 0.2738  (27.38%)
  Documente cu F1 > 0 : 803 / 1114


In [8]:
N_PREVIEW = 20

def make_html_table(df_preview):
    rows = ''
    for _, r in df_preview.iterrows():
        color = '#c8f7c5' if r['f1'] >= 0.5 else '#fde8d8' if r['f1'] > 0 else '#f5f5f5'
        rows += f"""
        <tr>
            <td style='font-size:12px;color:#555;padding:6px 8px;'>{r['filename']}</td>
            <td style='color:#117a65;font-weight:500;padding:6px 8px;'>{r['keywords_extracted']}</td>
            <td style='color:#555;padding:6px 8px;'>{r['keywords_original']}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['precision']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['recall']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;background:{color};font-weight:bold;'>{r['f1']:.2f}</td>
        </tr>"""

    html = f"""
    <h3 style='font-family:sans-serif;'>🔍 TextRank — Primele {N_PREVIEW} articole</h3>
    <table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px;'>
        <thead>
            <tr style='background:#117a65;color:white;'>
                <th style='padding:8px;text-align:left;'>Fișier</th>
                <th style='padding:8px;text-align:left;'>Keywords extrase (TextRank)</th>
                <th style='padding:8px;text-align:left;'>Keywords originale</th>
                <th style='padding:8px;text-align:center;'>Precision</th>
                <th style='padding:8px;text-align:center;'>Recall</th>
                <th style='padding:8px;text-align:center;'>F1</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style='font-family:sans-serif;font-size:12px;color:#888;margin-top:8px;'>
        🟢 F1 ≥ 0.5 &nbsp;|&nbsp; 🟠 F1 > 0 &nbsp;|&nbsp; ⚪ F1 = 0
    </p>
    """
    return html

display(HTML(make_html_table(results_df.head(N_PREVIEW))))


Fișier,Keywords extrase (TextRank),Keywords originale,Precision,Recall,F1
RMJ_2017_3_Art-08.pdf,pacienți; polipilor; spitalul clinic nr. 1 c.f. witting bucurești; unități spitalicești; spital,tumori; maligne; studiu; retrospectiv; endoscopic,0.00,0.00,0.00
RMR_Nr-3_2011_Art-6.pdf,testul hawkins; testul patte; testul jobe; impingement neer; pacienți,sindrom impingement; umăr; teste clinice; regiune deltoidiana,0.40,0.50,0.44
RJP_2017_2_RO_Art-05.pdf,tratamentul; intracraniană; complex; 10 ani; 5,meduloblastom; copil,0.00,0.00,0.00
RJP_2012_1_Art-09.pdf,tratamentul substitutiv; gh; arginină; copil; hipoglicemiilor,hipoglicemie persistentă; deﬁcit gh; tratament substitutiv,0.60,1.00,0.75
RJP_2015_4_RO_Art-07.pdf,copil; bolnavilor; progresele; ultimele decade,fibroză chistică; etică; screening; terapie genică; transplant; fertilitate,0.00,0.00,0.00
RJS_2014_3_Art-09.pdf,hemangioamelor; gâtului; copii; ultimii zece ani; primă,hemangiom; cavitate orală; perioral; propranolol; corticosteroizi; laser; intralezional; scleroterapie,0.40,0.25,0.31
RJP_2012_2_Art-13.pdf,istoricul medical; pacient; tumoră; 1-2 litri; 14 ani,hidronefroză gigantă; hipertensiune; copil,0.20,0.33,0.25
RJP_2012_2_Art-07.pdf,prezența wheezing; debut; diagnosticul de wheezing recurent postbroșiolitic neînsoțit; factori de risc pentru wheezing; principalilor,wheezing recurent; copil; factori de risc; atopie,0.40,0.50,0.44
RJP_2011_1_Art-01.pdf,diagnosticul alergiei alimentare; mastocitele intestinale; ige serice; triptazei; anamneză,alergie alimentară; diagnostic; copil,0.20,0.33,0.25
RMJ_2019_1_Art-11.pdf,th; pacienți; pacienții; dz2; sanamed hospital,diabet zaharat tip 2; tiroidită; autoimun; corelație,0.20,0.25,0.22
